# Orca Nano — QLoRA Fine-Tune v3 (Kaggle, final hardened version)

**This version exists because the last two Kaggle runs both lost a fully-trained model** — once to a disk-space crash during GGUF export, once to a silent session death mid-run with no checkpoint to recover from. Every fix below is baked in from the start, not patched in after a crash.

## Setup, before running any cell
1. **Enable GPU**: right sidebar → Settings → Accelerator → **GPU T4 x2** (needs phone verification once, if not already done).
2. **Enable internet**: right sidebar → Settings → Internet → **On** (off by default).
3. **Attach your training data**: right sidebar → **Add Input** → attach your `orca-nano-training-data` dataset (containing `orca_nano_llama3_train_v3_safety.jsonl` and `orca_llama3_eval.jsonl`).
4. **Save your work as you go**: click **Save Version → Save & Run All (Commit)** after training succeeds, even before trying the export cells. A committed version is the ONLY thing Kaggle's API/Output tab can serve — an uncommitted Draft Session's files disappear if the session dies.

## Every bug from the last two rounds, fixed proactively
- **Disk-space crash** (`OSError: Not enough free space`, `RuntimeError: Unsloth: Failed saving locally`): `/kaggle/working/` has only a 19.5GB quota, and a merged 16-bit 7B model + F16 GGUF intermediate blow past that even after cleanup. Fixed by doing the merge + GGUF conversion in `/tmp` (separate, larger disk), then copying only the final quantized file back to `/kaggle/working/`.
- **Silent session death losing the trained model**: fixed two ways — (a) checkpoints saved to `/kaggle/working/checkpoints` every 50 steps during training, with automatic resume-from-checkpoint if you have to re-run the training cell after a restart; (b) the LoRA adapter is saved to `/kaggle/working/adapter` in its own cell **immediately after training finishes**, before merge/export even starts — so a crash during merge/export no longer costs you the trained weights, just a re-run of one cell.
- **`AttributeError: 'int' object has no attribute 'mean'`**: known Unsloth/Transformers bug ([unslothai/unsloth#3769](https://github.com/unslothai/unsloth/issues/3769)) — `average_tokens_across_devices=False` is in `TrainingArguments` from the start.
- **Slow install**: plain PyPI `unsloth`, not `git+https://...` (which triggers a slow build-from-source).
- **Xet download stall**: disabled before any import.
- **Glob path/case mismatch**: Unsloth's GGUF output folder naming varies run to run (`gguf/` vs `gguf_gguf/`, case varies) — search is recursive and case-insensitive.

Training config unchanged: rank 16 LoRA, 2050/108 dataset (full nano distillation batch + 150 targeted `honesty_hedging` examples), 2 epochs — v1's validation loss rose after step 100 at 3 epochs, a real overfitting signal on this dataset size.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

# Plain PyPI install — no git+https source, no build-from-source step.
!pip install -q unsloth trl transformers datasets peft bitsandbytes accelerate

## Find your uploaded training data

Searches recursively under `/kaggle/input/` so it doesn't matter what your dataset was named.

In [ ]:
import glob

train_matches = glob.glob('/kaggle/input/**/orca_nano_llama3_train_v3_safety.jsonl', recursive=True)
eval_matches  = glob.glob('/kaggle/input/**/orca_llama3_eval.jsonl', recursive=True)

print('Train file found:', train_matches)
print('Eval file found:', eval_matches)

if not train_matches:
    raise FileNotFoundError(
        "orca_nano_llama3_train_v3_safety.jsonl not found under /kaggle/input/. "
        "Make sure the dataset is attached via 'Add Input' in the right sidebar."
    )

train_path = train_matches[0]
eval_path = eval_matches[0] if eval_matches else None

In [ ]:
import json

def load_jsonl(path):
    lines = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    lines.append(json.loads(line))
                except Exception:
                    pass
    return lines

raw_train = load_jsonl(train_path)
raw_eval  = load_jsonl(eval_path) if eval_path else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit) + attach LoRA

Rank 16 — sized for a free-tier GPU's VRAM, not the 128-rank A100 cloud preset.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
base_model = "unsloth/Qwen2.5-7B-Instruct"  # exact case matters on Hugging Face

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import Dataset

def format_conv(ex):
    turns = ex.get("conversations", ex.get("text"))
    if isinstance(turns, str):
        return turns  # already-formatted llama3 text field
    parts = []
    for t in turns:
        role = t.get("role", "")
        val  = t.get("value", "")
        if role == "system":
            parts.append(f"<|start_header_id|>system<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "human":
            parts.append(f"<|start_header_id|>user<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "gpt":
            parts.append(f"<|start_header_id|>assistant<|end_header_id|>\n\n{val}<|eot_id|>")
    return "".join(parts)

# The formatter.py output already has a 'text' field per example (llama3 format) — use it directly if present.
train_ds = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_train])
eval_ds  = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_eval])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Batch size 2 + grad accumulation 4 (effective batch 8), 2 epochs.
`average_tokens_across_devices=False` prevents the known Unsloth/Transformers crash.

**No mid-training checkpointing** (`save_strategy="no"`) — an earlier version of
this notebook tried checkpointing every 50 steps to survive a session death
mid-run, but that hit a real Kaggle-side bug: `PicklingError: Can't pickle
<class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as
trl.trainer.sft_config.SFTConfig` — a version mismatch between the installed
`trl`/`transformers` and Unsloth's compiled trainer cache, triggered
specifically by the checkpoint-save path, not by training itself. Since the
very next cell saves the LoRA adapter immediately after `trainer.train()`
completes, that's the real safety net — no mid-training checkpoint is needed.
If the session dies mid-training, you'll need to re-run this cell from
scratch, but at least it won't crash on its own.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=100,
        save_strategy="no",
        output_dir="/kaggle/working/output",
        eval_strategy="steps",
        report_to="none",
        average_tokens_across_devices=False,
    ),
)

print("[train] starting QLoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Save the LoRA adapter immediately (before merge/export)

This is the single most important cell in this notebook. The adapter is small
(~100-200MB) and fast to save. Once this succeeds, **the trained weights are
safe** — even if the merge/GGUF-export step below crashes, you keep this and
can re-run just the merge/export against it later without retraining.

In [ ]:
adapter_dir = "/kaggle/working/adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"[adapter] saved to {adapter_dir} — trained weights are now safe on disk.")
!ls -la {adapter_dir}

## Merge LoRA + export GGUF (done in /tmp, not /kaggle/working)

**Why /tmp**: `/kaggle/working/` has a hard 19.5GB quota. The merged 16-bit
7B model plus the intermediate F16 GGUF file exceed that even after deleting
the merged folder — this is exactly what crashed the last Kaggle run
(`OSError: Not enough free space`). `/tmp` is a separate, much larger scratch
disk, so both intermediates fit comfortably. Only the final quantized
`.gguf` file gets copied back to `/kaggle/working/` afterward.

In [ ]:
import shutil

# Clean up any stale /tmp output from a previous failed attempt in this session.
shutil.rmtree("/tmp/merged", ignore_errors=True)
shutil.rmtree("/tmp/gguf", ignore_errors=True)

print("[merge] merging LoRA adapters (in /tmp)...")
model.save_pretrained_merged("/tmp/merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to /tmp/merged")

print("[gguf] converting to GGUF q4_k_m (in /tmp)...")
model.save_pretrained_gguf("/tmp/gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved under /tmp")

## Copy the final GGUF back to /kaggle/working/

Recursive + case-insensitive search — Unsloth's output folder naming has
varied between runs (e.g. `gguf/` vs `gguf_gguf/`), so this searches broadly
under `/tmp` instead of assuming one exact path.

In [ ]:
import glob, shutil, os

candidates = [f for f in glob.glob('/tmp/**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found in /tmp:', candidates)

if candidates:
    source_path = candidates[0]
    filename = os.path.basename(source_path)
    dest_path = f'/kaggle/working/{filename}'
    shutil.copy(source_path, dest_path)
    print(f"[export] copied to {dest_path}")
    print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")
    print("Once the commit finishes, go to this notebook's 'Output' tab (or Data tab")
    print("of the committed version) and download the .gguf file from there.")
else:
    print('No GGUF file found under /tmp — check the [gguf] cell above for errors.')
    print('If training + adapter save both succeeded, your trained weights are still')
    print('safe in /kaggle/working/adapter — you can retry just this export cell.')